Импортируем необходимые библиотеки.

In [2]:
import os
import torch
import glob
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import torch.nn as nn
from transformers import AutoImageProcessor, AutoModelForImageClassification, TrainingArguments, Trainer
from datasets import Dataset as HFDataset
import numpy as np
from sklearn.metrics import accuracy_score, classification_report
from collections import defaultdict
import wandb
import random
from tqdm import tqdm, tqdm_notebook
from torch.utils.data import Subset

np.random.seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"{device=}")

device=device(type='cuda')


**1.1 Transfer learning**

1.1.1 Выберите одну из трансформерных классификационных моделей, доступных на HF. Обоснуйте свой выбор модели. Возможные критерии для выбора - релевантность обучающих данных, точность классификации, размер модели, требуемые ресурсы для обучения.

Возьмем модель microsoft/beit-base-patch16-224, так как:
- она обучена на ImageNet-21k (14 million images, 21,841 classes)
- выполнен fine-tuning на ImageNet 2012 (1 million images, 1,000 classes)

То есть модель обучена на огромном количестве изображений различных классов.
- модель относительно небольшая (87M params), значит будет обучаться довольно быстро и хватит ресурсов gpu.
- Совместима с Hugging Face Transformers (AutoModelForImageClassification)

In [3]:
from transformers import AutoImageProcessor, AutoModelForImageClassification
model_name = "microsoft/beit-base-patch16-224"
processor = AutoImageProcessor.from_pretrained(model_name)

preprocessor_config.json:   0%|          | 0.00/276 [00:00<?, ?B/s]

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


1.1.2 Для выбранной модели выполните хотя бы один вариант transfer learning для решения поставленной задачи. Используя инструменты HF и библиотеки transformers напишите код для подгрузки и обработки данных, обучения и инференса модели, логирования результатов. Оцените полученное качество классификации, сравните с результатами, полученными в HW2. 


Напишем классы SimpsonsTrainDataset и SimpsonsTestDataset для создания датасетов подходящего формата. Так как в структура train_dataset и test_dataset различается, придется написать 2 разных класса.

In [4]:
class SimpsonsTrainDataset(Dataset):
    def __init__(self, data_dir, processor, split="train"):
        self.data_dir = data_dir
        self.processor = processor
        self.split = split

        self.samples = []
        self.class_to_idx = {}
        self.idx_to_class = {}
        
        classes = sorted([d for d in os.listdir(data_dir) if os.path.isdir(os.path.join(data_dir, d))])
        self.class_to_idx = {cls_name: idx for idx, cls_name in enumerate(classes)}
        self.idx_to_class = {idx: cls_name for cls_name, idx in self.class_to_idx.items()}
        
        for class_name in classes:
            class_dir = os.path.join(data_dir, class_name)
            for img_name in os.listdir(class_dir):
                if img_name.lower().endswith(('.png', '.jpg', '.jpeg')):
                    self.samples.append((os.path.join(class_dir, img_name), self.class_to_idx[class_name]))
    
    def __len__(self):
        return len(self.samples)
    
    def __getitem__(self, idx):
        img_path, label = self.samples[idx]
        image = Image.open(img_path).convert('RGB')
        inputs = self.processor(image, return_tensors="pt")
        
        return {'pixel_values': inputs['pixel_values'].squeeze(), 'labels': torch.tensor(label, dtype=torch.long)}

In [5]:
class SimpsonsTestDataset(Dataset):
    def __init__(self, test_dir, processor, class_to_idx):
        self.test_dir = test_dir
        self.processor = processor
        self.class_to_idx = class_to_idx

        self.samples = []
        test_images = glob.glob(os.path.join(test_dir, "*.jpg")) + \
                     glob.glob(os.path.join(test_dir, "*.png")) + \
                     glob.glob(os.path.join(test_dir, "*.jpeg"))
        
        for img_path in test_images:
            filename = os.path.basename(img_path)
            class_name = '_'.join(filename.split('_')[:-1])
            #print(class_name)
            
            if class_name in self.class_to_idx:
                label = self.class_to_idx[class_name]
                self.samples.append((img_path, label, filename))
            else:
                print(f"Unknown class {class_name} in file {filename}")
                continue
    
    def __len__(self):
        return len(self.samples)
    
    def __getitem__(self, idx):
        img_path, label, filename = self.samples[idx]
        image = Image.open(img_path).convert('RGB')
        inputs = self.processor(image, return_tensors="pt")
        
        return {
            'pixel_values': inputs['pixel_values'].squeeze(), 'labels': torch.tensor(label, dtype=torch.long), 'filename': filename}


In [6]:
num_classes = 20
batch_size = 64
img_size = 224

processor.size = img_size

train_dataset = SimpsonsTrainDataset("/kaggle/input/simpsons/train_dataset", processor)
test_dataset = SimpsonsTestDataset("/kaggle/input/simpsons/test_dataset", processor, train_dataset.class_to_idx)

print(f"Train samples: {len(train_dataset)}")
print(f"Test samples: {len(test_dataset)}")
print(f"Classes: {train_dataset.idx_to_class}")

Train samples: 19548
Test samples: 990
Classes: {0: 'abraham_grampa_simpson', 1: 'apu_nahasapeemapetilon', 2: 'bart_simpson', 3: 'charles_montgomery_burns', 4: 'chief_wiggum', 5: 'comic_book_guy', 6: 'edna_krabappel', 7: 'homer_simpson', 8: 'kent_brockman', 9: 'krusty_the_clown', 10: 'lenny_leonard', 11: 'lisa_simpson', 12: 'marge_simpson', 13: 'mayor_quimby', 14: 'milhouse_van_houten', 15: 'moe_szyslak', 16: 'ned_flanders', 17: 'nelson_muntz', 18: 'principal_skinner', 19: 'sideshow_bob'}


из HW2: Датасет сильно несбалансирован по классам. Например, класс mayor_quimby составляет примерно 0.1 от класса homer_simpson, что при обучении модели будет играть роль. Поэтому из всего тренировочного датасета возьмем только по 200-600 изображений каждого класса, чтобы распределение было чуть равномернее, дабы не создавать ситуации, когда модель хорошо определяет только самые объемные по количеству картинок классы исходного датасета.

В HW2 был проанализирован датасет и было сказано, что он сильно несбалансированный по классам. Применим тот же метод отбора картинок каждого класса, что и тогда. Будем брать все изображения класса, если их меньше 600, будем брать рандомное количество от 500 до 600 изображений в противном случае. Так наши данные будут более сбалансированные.

In [7]:
class_indices = defaultdict(list)

for idx in range(len(train_dataset)):
    sample = train_dataset[idx]
    target = sample['labels'].item()  
    class_indices[target].append(idx)

selected_indices = []

for cls, indices in class_indices.items():
    count = len(indices)
    class_name = train_dataset.idx_to_class[cls]
    
    if count < 600:
        selected_indices.extend(indices)
    else:
        sample_size = random.randint(500, 600)
        sampled = random.sample(indices, sample_size)
        selected_indices.extend(sampled)
random.shuffle(selected_indices)

balanced_train = Subset(train_dataset, selected_indices)
print(f"Final size of train dataset is: {len(balanced_train)}")

Final size of train dataset is: 10010


Теперь тренировочных данных стало примерно в 10 раз больше тестовых, также улучшился баланс классов.

In [8]:
model = AutoModelForImageClassification.from_pretrained(model_name,
                                                        num_labels=num_classes,
                                                        ignore_mismatched_sizes=True)
#print(model)

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/350M [00:00<?, ?B/s]

Some weights of BeitForImageClassification were not initialized from the model checkpoint at microsoft/beit-base-patch16-224 and are newly initialized because the shapes did not match:
- classifier.bias: found shape torch.Size([1000]) in the checkpoint and torch.Size([20]) in the model instantiated
- classifier.weight: found shape torch.Size([1000, 768]) in the checkpoint and torch.Size([20, 768]) in the model instantiated
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [9]:
print(model)

BeitForImageClassification(
  (beit): BeitModel(
    (embeddings): BeitEmbeddings(
      (patch_embeddings): BeitPatchEmbeddings(
        (projection): Conv2d(3, 768, kernel_size=(16, 16), stride=(16, 16))
      )
      (dropout): Dropout(p=0.0, inplace=False)
    )
    (encoder): BeitEncoder(
      (layer): ModuleList(
        (0): BeitLayer(
          (attention): BeitAttention(
            (attention): BeitSdpaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=False)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.0, inplace=False)
              (relative_position_bias): BeitRelativePositionBias()
            )
            (output): BeitSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.0, inplace=False)
            )
          )
          

In [10]:
for name, param in model.named_parameters():
        if "classifier" not in name and "head" not in name:
            param.requires_grad = False
        else:
            param.requires_grad = True

trainable_params = [param for param in model.parameters() if param.requires_grad]

Напишем функцию для вычисления accuracy и f1-score.

In [11]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    accuracy = accuracy_score(labels, predictions)

    report = classification_report(
        labels, 
        predictions, 
        output_dict=True, 
        zero_division=0
    )
    
    f1 = report['weighted avg']['f1-score']

    unique_preds = np.unique(predictions)
    unique_labels = np.unique(labels)
    
    if len(unique_preds) < len(unique_labels):
        missing_classes = set(unique_labels) - set(unique_preds)
        print(f"No predictions for classes: {missing_classes}")
    
    return {"accuracy": accuracy, "f1": f1}

In [12]:
from transformers import TrainerCallback
class ProgressCallback(TrainerCallback):
    def on_log(self, args, state, control, logs=None, **kwargs):
        if state.is_local_process_zero:
            if 'loss' in logs:
                print(f"Step {state.global_step}: loss = {logs['loss']:.4f}")
            if 'eval_loss' in logs:
                print(f"Step {state.global_step}: eval_loss = {logs['eval_loss']:.4f}, eval_accuracy = {logs.get('eval_accuracy', 0):.4f}")

Будем использовать инструменты transformers для обучения и инференса модели 

In [13]:
batch_size = 64
training_args = TrainingArguments(output_dir="./simpsons_classifier",
    num_train_epochs=10,
    per_device_train_batch_size=batch_size,
    per_device_eval_batch_size=batch_size,
    learning_rate=0.001,
    warmup_steps=100,
    logging_steps=100,
    eval_steps=100,
    save_steps=200,
    eval_strategy="steps",
    save_strategy="steps",
    disable_tqdm=False,         
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    greater_is_better=True,
    remove_unused_columns=False,
    dataloader_pin_memory=False,
    logging_dir="./logs",
    report_to="none")

trainer = Trainer(model=model,
                args=training_args,
                train_dataset=balanced_train,
                eval_dataset=test_dataset,
                compute_metrics=compute_metrics,
                processing_class=processor,
                callbacks=[ProgressCallback()])

In [14]:
import warnings
def evaluate_model(trainer, dataset):
    predictions = trainer.predict(dataset)
    pred_labels = np.argmax(predictions.predictions, axis=1)
    true_labels = predictions.label_ids

    accuracy = accuracy_score(true_labels, pred_labels)
    print(f"Accuracy on the test samples: {accuracy:.4f}")

    class_names = [train_dataset.idx_to_class[i] for i in range(len(train_dataset.class_to_idx))]
    
    print(f"Classification Report:")
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        report = classification_report(true_labels, pred_labels, target_names=class_names, zero_division=0,digits=4)
    print(report)

    unique_preds, pred_counts = np.unique(pred_labels, return_counts=True)
    unique_true, true_counts = np.unique(true_labels, return_counts=True)
    
    print(f"Prediction Distribution:")
    for i, class_name in enumerate(class_names):
        pred_count = pred_counts[unique_preds == i][0] if i in unique_preds else 0
        true_count = true_counts[unique_true == i][0] if i in unique_true else 0
        print(f"  {class_name}: {pred_count}/{true_count} predicted")
    
    return accuracy, pred_labels, true_labels

In [15]:
print(device)

cuda


Обучение

In [16]:
model.to(device)
train_results = trainer.train()
trainer.save_model("./simpsons_microsoft")
processor.save_pretrained("./simpsons_microsoft")
print("Model saved to ./simpsons_microsoft")

Step,Training Loss,Validation Loss,Accuracy,F1
100,2.399700,1.338385,0.698990,0.694401
200,1.026000,0.717477,0.844444,0.844998
300,0.666900,0.530409,0.881818,0.882440
400,0.520200,0.442623,0.890909,0.890680
500,0.450700,0.369689,0.913131,0.913168
600,0.396700,0.350854,0.919192,0.919051
700,0.346900,0.336171,0.924242,0.924179
800,0.339000,0.306894,0.929293,0.929338
900,0.299800,0.292595,0.923232,0.923260
1000,0.285500,0.288085,0.923232,0.923089


Step 100: loss = 2.3997
Step 100: eval_loss = 1.3384, eval_accuracy = 0.6990
Step 200: loss = 1.0260
Step 200: eval_loss = 0.7175, eval_accuracy = 0.8444
Step 300: loss = 0.6669
Step 300: eval_loss = 0.5304, eval_accuracy = 0.8818
Step 400: loss = 0.5202
Step 400: eval_loss = 0.4426, eval_accuracy = 0.8909
Step 500: loss = 0.4507
Step 500: eval_loss = 0.3697, eval_accuracy = 0.9131
Step 600: loss = 0.3967
Step 600: eval_loss = 0.3509, eval_accuracy = 0.9192
Step 700: loss = 0.3469
Step 700: eval_loss = 0.3362, eval_accuracy = 0.9242
Step 800: loss = 0.3390
Step 800: eval_loss = 0.3069, eval_accuracy = 0.9293
Step 900: loss = 0.2998
Step 900: eval_loss = 0.2926, eval_accuracy = 0.9232
Step 1000: loss = 0.2855
Step 1000: eval_loss = 0.2881, eval_accuracy = 0.9232
Step 1100: loss = 0.2788
Step 1100: eval_loss = 0.2776, eval_accuracy = 0.9343
Step 1200: loss = 0.2636
Step 1200: eval_loss = 0.2650, eval_accuracy = 0.9364
Step 1300: loss = 0.2541
Step 1300: eval_loss = 0.2641, eval_accuracy 

Инференс

In [17]:
test_accuracy_mobilenet, test_preds_mobilenet, test_true = evaluate_model(trainer, test_dataset)

Accuracy on the test samples: 0.9404
Classification Report:
                          precision    recall  f1-score   support

  abraham_grampa_simpson     1.0000    0.8542    0.9213        48
  apu_nahasapeemapetilon     1.0000    1.0000    1.0000        50
            bart_simpson     1.0000    0.8600    0.9247        50
charles_montgomery_burns     0.8679    0.9583    0.9109        48
            chief_wiggum     0.9792    0.9400    0.9592        50
          comic_book_guy     0.8545    0.9592    0.9038        49
          edna_krabappel     0.9057    0.9600    0.9320        50
           homer_simpson     0.9556    0.8600    0.9053        50
           kent_brockman     0.9423    0.9800    0.9608        50
        krusty_the_clown     0.9245    0.9800    0.9515        50
           lenny_leonard     1.0000    0.9600    0.9796        50
            lisa_simpson     0.9592    0.9400    0.9495        50
           marge_simpson     0.9400    0.9400    0.9400        50
            may

В HW2 лучшими моделями были AlexNet после transfer learning и CNNBatchNorm, которые показывали accuracy от 0.97 до 1 на тестовых данных при разных размерах батча. 
Модель microsoft/beit-base-patch16-224 имеет accuracy 0.94 что не сильно хуже, чем у тех двух. Скорее всего качество ниже, так как CNNBatchNorm обучалась с самого начала, она заточена на классификацию симпсонов, в то время как у этой модели мы используем ее backbone и переделываем только последний слой (классификационный). В случае с AlexNet, который при том же transfer learning показал результат лучше, архитектура могла сыграть роль. По экспериментам в дз2 и здесь видно, что сверточные нейросети сработали лучше для задачи классификации Симпсонов, нежели трансформерная.

In [ ]:
!zip -r videos24_31.zip /kaggle/working/simpsons_classifier

  adding: kaggle/working/simpsons_classifier/ (stored 0%)
  adding: kaggle/working/simpsons_classifier/checkpoint-800/ (stored 0%)
  adding: kaggle/working/simpsons_classifier/checkpoint-800/preprocessor_config.json (deflated 50%)
  adding: kaggle/working/simpsons_classifier/checkpoint-800/trainer_state.json (deflated 74%)
  adding: kaggle/working/simpsons_classifier/checkpoint-800/model.safetensors (deflated 41%)
  adding: kaggle/working/simpsons_classifier/checkpoint-800/rng_state.pth (deflated 25%)
  adding: kaggle/working/simpsons_classifier/checkpoint-800/optimizer.pt (deflated 10%)
  adding: kaggle/working/simpsons_classifier/checkpoint-800/scheduler.pt (deflated 55%)
  adding: kaggle/working/simpsons_classifier/checkpoint-800/training_args.bin (deflated 52%)
  adding: kaggle/working/simpsons_classifier/checkpoint-800/config.json (deflated 65%)
  adding: kaggle/working/simpsons_classifier/checkpoint-1000/ (stored 0%)
  adding: kaggle/working/simpsons_classifier/checkpoint-1000/pr